In [0]:
from datetime import datetime
import uuid

pipeline_name = "orders_pipeline"

batch_id = "batch_003"
restart_from = "SILVER"   # BRONZE / SILVER / GOLD

restart_from = restart_from.upper().strip()

allowed_stages = ["BRONZE", "SILVER", "GOLD"]

if restart_from not in allowed_stages:
    raise ValueError(
        f"Invalid restart_from='{restart_from}'. "
        f"Allowed values: {allowed_stages}"
    )


# ---------------------------------------------------------
# 0A. Find current RUNNING run
# ---------------------------------------------------------

running_runs = spark.sql(f"""
    SELECT run_id, batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = '{pipeline_name}'
      AND status = 'RUNNING'
""").collect()

if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, "
        f"found {len(running_runs)}."
    )

old_run_id = running_runs[0]["run_id"]
old_batch_id = running_runs[0]["batch_id"]

if old_batch_id != batch_id:
    raise ValueError(
        f"RUNNING run belongs to {old_batch_id}, "
        f"but retry requested for {batch_id}."
    )


# ---------------------------------------------------------
# 0B. Close old run as FAILED
# ---------------------------------------------------------

error_message = (
    f"Run manually rejected for recovery. "
    f"Retry requested from {restart_from}."
)

spark.sql(f"""
    UPDATE workspace.control.etl_run_log
    SET
        status = 'FAILED',
        end_timestamp = CURRENT_TIMESTAMP(),
        error_message = '{error_message}'
    WHERE run_id = '{old_run_id}'
      AND status = 'RUNNING'
""")

print(f"Previous run marked FAILED: {old_run_id}")


# ---------------------------------------------------------
# 1. Validate input
# ---------------------------------------------------------

restart_from = restart_from.upper().strip()

allowed_stages = ["BRONZE", "SILVER", "GOLD"]

if restart_from not in allowed_stages:
    raise ValueError(
        f"Invalid restart_from='{restart_from}'. "
        f"Allowed values: {allowed_stages}"
    )


# ---------------------------------------------------------
# 2. Guardrail: no existing RUNNING run
# ---------------------------------------------------------

running_runs = spark.sql(f"""
    SELECT run_id, batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = '{pipeline_name}'
      AND status = 'RUNNING'
""").collect()

if len(running_runs) != 0:
    raise ValueError(
        f"Cannot create retry run. "
        f"Found {len(running_runs)} existing RUNNING run(s)."
    )


# ---------------------------------------------------------
# 3. Check that batch exists
# ---------------------------------------------------------

landing_rows = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM workspace.landing.orders
    WHERE batch_id = '{batch_id}'
""").first()["cnt"]

if landing_rows == 0:
    raise ValueError(
        f"Batch {batch_id} not found in Landing."
    )


# ---------------------------------------------------------
# 4. Load counts from known-good layers
# ---------------------------------------------------------

bronze_rows = None
silver_rows = None
rejected_rows = None


if restart_from in ["SILVER", "GOLD"]:

    bronze_rows = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.bronze.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    if bronze_rows == 0:
        raise ValueError(
            f"Cannot restart from {restart_from}. "
            f"Batch {batch_id} not found in Bronze."
        )


if restart_from == "GOLD":

    silver_rows = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.silver.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    if silver_rows == 0:
        raise ValueError(
            f"Cannot restart from GOLD. "
            f"Batch {batch_id} not found in Silver."
        )

    rejected_rows = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.control.rejected_orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]


# ---------------------------------------------------------
# 5. Prepare SQL values
# ---------------------------------------------------------

def sql_value(value):
    return "NULL" if value is None else str(value)


run_id = str(uuid.uuid4())
run_start = datetime.now()

landing_sql = sql_value(landing_rows)
bronze_sql = sql_value(bronze_rows)
silver_sql = sql_value(silver_rows)
rejected_sql = sql_value(rejected_rows)


# ---------------------------------------------------------
# 6. Create retry RUNNING audit row
# ---------------------------------------------------------

spark.sql(f"""
INSERT INTO workspace.control.etl_run_log (
    run_id,
    batch_id,
    pipeline_name,
    start_timestamp,
    end_timestamp,
    landing_rows,
    bronze_rows,
    silver_rows,
    gold_inserted,
    gold_updated,
    rejected_rows,
    status,
    error_message
)
VALUES (
    '{run_id}',
    '{batch_id}',
    '{pipeline_name}',
    TIMESTAMP '{run_start}',
    NULL,
    {landing_sql},
    {bronze_sql},
    {silver_sql},
    NULL,
    NULL,
    {rejected_sql},
    'RUNNING',
    NULL
)
""")


# ---------------------------------------------------------
# 7. Summary
# ---------------------------------------------------------

print("--------------------------------------------------")
print("RETRY RUN CREATED")
print("--------------------------------------------------")
print(f"Run ID:         {run_id}")
print(f"Batch ID:       {batch_id}")
print(f"Restart from:   {restart_from}")
print(f"Landing rows:   {landing_rows}")
print(f"Bronze rows:    {bronze_rows}")
print(f"Silver rows:    {silver_rows}")
print(f"Rejected rows:  {rejected_rows}")
print(f"Status:         RUNNING")
print("--------------------------------------------------")

Previous run marked FAILED: 8e423226-2c1e-481a-a55b-6b03339e138c
--------------------------------------------------
RETRY RUN CREATED
--------------------------------------------------
Run ID:         c803a791-ac92-4466-8a22-297efcb338d4
Batch ID:       batch_003
Restart from:   SILVER
Landing rows:   765
Bronze rows:    765
Silver rows:    None
Rejected rows:  None
Status:         RUNNING
--------------------------------------------------
